# HierarchicalDet Phase 0 setup (Kaggle GPU notebook)

Scope: environment install, DENTEX dataset download + 3-tier verification, and a single-image forward-pass smoke test of the inference pipeline (no pretrained HierarchicalDet weights are publicly released, so this only proves the pipeline runs mechanically -- Phase 2 covers actual training/eval).

**Before running**: Settings > Accelerator > GPU T4x2 (or P100). Settings > Internet > On. Settings > Persistence > "Variables and files". Settings > Environment > "Pin to original environment".

**Only 3 code cells, deliberately.** Earlier versions of this notebook had ~12 small cells, and repeatedly ran into a real workflow problem: pasting an updated cell as a *new* cell (rather than replacing the old one) left a stale duplicate further down that kept silently executing instead of the fix. Consolidating everything into 3 cells makes that class of mistake much harder to make -- if something needs updating, there's only one place per stage to look.

**Every cell is safe to re-run, including after a session restart or Run All.** Kaggle's "pin to original environment" reverts to the base image's default packages on every fresh session -- anything pip-installed outside `/kaggle/working` does NOT survive a restart, only files under `/kaggle/working` do -- so Cell 1 always reinstalls (fast, since pip no-ops on already-satisfied packages).

**Disk space**: Kaggle's `/kaggle/working` quota (~20GB) can't hold the raw downloaded DENTEX dataset (11.8GB) plus a full extraction of the largest archive (`training_data.zip`, 10.9GB compressed) at the same time -- confirmed on a real run ("OSError: No space left on device", including for something as small as creating an empty directory once the disk filled up completely). Cell 2 never extracts `training_data.zip`/`test_data.zip` (reads their stats directly via `zipfile`, no extraction needed) and deletes both immediately after, freeing ~11.6GB. This means a future session will need to re-download those two archives again even with file persistence on -- unavoidable given the disk quota, not a bug.

Everything here -- the Pillow/legacy-PIL-constant issue, the PIL._util.is_directory issue, the pycocotools `_mask` extension, the exact dataset paths, the Swin-B checkpoint size mismatch, the disk-space handling -- has been verified end-to-end against a local Python 3.9/3.12 environment and real downloaded data before ever being run on Kaggle. If something still fails here, it's a genuinely new issue, not a repeat of one already chased down; paste the full traceback rather than just the last few lines, since some earlier bugs here only became clear from output further up (shape-mismatch and disk-space warnings in particular are easy to miss in a long log).

## Cell 1 -- environment setup

GPU check, clone/update the repo, install dependencies, fix the bundled `pycocotools` extension, verify the full import chain.

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in notebook Settings before proceeding.'

# Clone (or update) the repo. Idempotent: safe to re-run after a restart --
# pulls latest instead of re-cloning if /kaggle/working/repo already exists.
REPO_URL = 'https://github.com/christopherh-88/HierarchicalDet.git'
%cd /kaggle/working
import os
if os.path.isdir('repo/.git'):
    %cd repo
    !git pull
else:
    !git clone $REPO_URL repo
    %cd repo

# Install dependencies. Always re-run this on a fresh session -- pip installs
# outside /kaggle/working don't persist across restarts here.
#
# Pillow note: this old detectron2 snapshot uses PIL.Image.LINEAR (removed in
# Pillow 10) and relies on PIL._util.is_directory (also removed in newer
# Pillow, needed transitively via torchvision -> PIL.ImageFont). Both are now
# fixed at the source in the repo itself (detectron2/utils/env.py and
# several other files, redundantly) rather than via a pip pin -- pinning
# pillow<10 was tried and never reliably worked on Kaggle across multiple
# attempts (wrong interpreter for `!pip install`, then PEP-668 blocking
# installs silently, then a reinstall reporting success while doing
# nothing). Nothing needs to happen here for Pillow anymore.
import sys, subprocess
print('sys.executable:', sys.executable)

def pip_install(args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + args
    try:
        subprocess.run(cmd + ['--break-system-packages'], check=True,
                        capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        if 'no such option' in (e.stderr or '').lower() or 'unrecognized arguments' in (e.stderr or '').lower():
            # older pip without PEP 668 support -- retry without the flag
            subprocess.run(cmd, check=True, capture_output=True, text=True)
        else:
            print(e.stdout)
            print(e.stderr)
            raise

# Exclude torch/torchvision/torchaudio -- Kaggle ships a CUDA-matched build
# already, don't clobber it.
subprocess.run(
    "grep -v -i -E '^(torch|torchvision|torchaudio)$' requirements.txt > /tmp/reqs_no_torch.txt",
    shell=True, check=True,
)
pip_install(['-r', '/tmp/reqs_no_torch.txt'])
pip_install(['huggingface_hub'])
print('Dependencies installed.')

# Bundled pycocotools/ (custom categories_1/2/3 3-tier format) ships without
# its compiled _mask Cython extension, and repo-root imports always shadow
# the pip-installed pycocotools package. Copy the compiled extension for this
# interpreter/platform from the pip install into the bundled package.
import site, glob, shutil
site_pkgs = site.getsitepackages()[0]
mask_so = glob.glob(f'{site_pkgs}/pycocotools/_mask*.so')
assert mask_so, f'no compiled _mask extension found under {site_pkgs}/pycocotools'
shutil.copy(mask_so[0], 'pycocotools/')
print('copied', mask_so[0])

# Verify the full import chain, same checks as local setup.sh
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from hierarchialdet import add_diffusiondet_config, DiffusionDetWithTTA
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper
from pycocotools.coco import COCO
import evaluator
print('All imports OK.')

## Cell 2 -- download DENTEX and verify all three annotation tiers

Verified 2026-07-14 by directly inspecting the real dataset (not guessed --
`training_data.zip` is 10.9GB, so its internal file listing was pulled via an
HTTP range request against just the zip's central directory, and the three
annotation JSONs were fetched via their own byte ranges, without downloading
the whole archive):

- `training_data.zip` contains **four** subfolders, not three:
  `quadrant/` (693 images, `train_quadrant.json`, flat `category_id`, 4 quadrant classes),
  `quadrant_enumeration/` (634 images, `train_quadrant_enumeration.json`, `category_id_1/2`, 4+8 classes),
  `quadrant-enumeration-disease/` (705 images, `train_quadrant_enumeration_disease.json`, `category_id_1/2/3`, 4+8+4 classes -- diagnosis classes are Impacted/Caries/Periapical Lesion/Deep Caries, matching the paper), and
  `unlabelled/` (1572 images, no annotations at all -- not documented in either README, not used by the default pipeline).
- `validation_data.zip` contains only images, at `validation_data/quadrant_enumeration_disease/xrays/val_N.png` (50 of them). The matching annotations are **not** in the zip -- they're the separate top-level `DENTEX/validation_triple.json` (50 images, 182 annotations, confirmed to reference these exact filenames).
- **`test_data.zip` uses a completely different, incompatible format**: per-image LabelMe polygon JSON files (`disease/label/test_N.json`) with Turkish-language diagnosis strings baked into a single `label` field (e.g. `"1-çürük-15"` = quadrant-diagnosis-tooth), images at `disease/input/test_N.png`. Not loadable by `register_coco_instances()` or `DiffusionDetDatasetMapper` as-is -- using the test split requires a LabelMe-to-COCO conversion script that doesn't exist yet anywhere in this codebase (Phase 1/2 scope; this cell only reports file counts).
- Category ID *numbering* is not consistent across files -- always read category names from each file's own `categories_N` list rather than assuming a fixed id-to-name mapping.

`training_data.zip` and `test_data.zip` are read directly (via `zipfile`, no extraction) and then **deleted** to free disk space -- see the disk-space note in the top markdown cell. Only `validation_data.zip` (small, 149.5MB) is extracted to disk, since that's where the smoke-test image comes from.

In [ ]:
import os, json, shutil, zipfile
from huggingface_hub import snapshot_download

RAW_DIR = '/kaggle/working/dentex_raw'
EXTRACT_DIR = '/kaggle/working/dentex_extracted'
os.makedirs(EXTRACT_DIR, exist_ok=True)

dataset_dir = snapshot_download(
    repo_id='ibrahimhamamci/DENTEX',
    repo_type='dataset',
    local_dir=RAW_DIR,
)
print('downloaded to', dataset_dir)

TRAINING_ZIP = os.path.join(dataset_dir, 'DENTEX/training_data.zip')
VALIDATION_ZIP = os.path.join(dataset_dir, 'DENTEX/validation_data.zip')
TEST_ZIP = os.path.join(dataset_dir, 'DENTEX/test_data.zip')

# Clean up any partial extraction left over from an earlier run that hit the
# disk-space error, to reclaim space before anything else runs.
partial_training_dir = os.path.join(EXTRACT_DIR, 'training_data')
if os.path.isdir(partial_training_dir):
    print('removing partial training_data extraction from a previous run to reclaim disk space...')
    shutil.rmtree(partial_training_dir)

# Only validation_data.zip gets fully extracted -- small, and it's where the
# smoke-test image comes from.
marker = os.path.join(EXTRACT_DIR, '.DENTEX_validation_data.zip.extracted')
if os.path.exists(marker):
    print('already extracted: validation_data.zip')
elif not os.path.exists(VALIDATION_ZIP):
    print('MISSING (not downloaded?):', VALIDATION_ZIP)
else:
    print('extracting', VALIDATION_ZIP, '...')
    with zipfile.ZipFile(VALIDATION_ZIP) as zf:
        zf.extractall(EXTRACT_DIR)
    open(marker, 'w').close()
    print('done: validation_data.zip')

VALIDATION_TRIPLE_PATH = os.path.join(dataset_dir, 'DENTEX', 'validation_triple.json')
VALIDATION_IMAGE_DIR = os.path.join(EXTRACT_DIR, 'validation_data', 'quadrant_enumeration_disease', 'xrays')

TRAINING_TIER_MEMBERS = {
    'quadrant (train)': ('training_data/quadrant/train_quadrant.json', 'training_data/quadrant/xrays/'),
    'quadrant-enumeration (train)': ('training_data/quadrant_enumeration/train_quadrant_enumeration.json', 'training_data/quadrant_enumeration/xrays/'),
    'quadrant-enumeration-diagnosis (train)': ('training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json', 'training_data/quadrant-enumeration-disease/xrays/'),
}
UNLABELLED_PREFIX = 'training_data/unlabelled/xrays/'

any_missing = False

if os.path.exists(TRAINING_ZIP):
    with zipfile.ZipFile(TRAINING_ZIP) as zf:
        names = zf.namelist()
        for tier, (json_member, img_prefix) in TRAINING_TIER_MEMBERS.items():
            if json_member not in names:
                print(tier, '-> MISSING member in zip:', json_member)
                any_missing = True
                continue
            d = json.loads(zf.read(json_member))
            n_images_in_zip = sum(1 for n in names if n.startswith(img_prefix) and n.endswith('.png'))
            print(tier, '->', json_member, '(inside training_data.zip, not extracted)')
            print('  images (JSON):', len(d.get('images', [])), ' images (zip entries):', n_images_in_zip, ' annotations:', len(d.get('annotations', [])))
            for k in d:
                if k.startswith('categor'):
                    print('  ', k, ':', [c['name'] for c in d[k]])
        n_unlabelled = sum(1 for n in names if n.startswith(UNLABELLED_PREFIX) and n.endswith('.png'))
        print('unlabelled xrays in zip (no annotations, not used by default pipeline):', n_unlabelled)
else:
    print('MISSING:', TRAINING_ZIP)
    any_missing = True

if os.path.exists(VALIDATION_TRIPLE_PATH):
    d = json.load(open(VALIDATION_TRIPLE_PATH))
    n_extracted = len([f for f in os.listdir(VALIDATION_IMAGE_DIR) if f.endswith('.png')]) if os.path.isdir(VALIDATION_IMAGE_DIR) else 0
    print('quadrant-enumeration-diagnosis (validation) ->', VALIDATION_TRIPLE_PATH)
    print('  images (JSON):', len(d.get('images', [])), ' images (extracted to disk):', n_extracted, ' annotations:', len(d.get('annotations', [])))
    for k in d:
        if k.startswith('categor'):
            print('  ', k, ':', [c['name'] for c in d[k]])
else:
    print('MISSING:', VALIDATION_TRIPLE_PATH)
    any_missing = True

print()
print('=== TEST SET: incompatible format, not COCO ===')
if os.path.exists(TEST_ZIP):
    with zipfile.ZipFile(TEST_ZIP) as zf:
        names = zf.namelist()
        n_test_images = sum(1 for n in names if n.startswith('disease/input/') and n.endswith('.png'))
        n_test_labels = sum(1 for n in names if n.startswith('disease/label/') and n.endswith('.json'))
    print(f'{n_test_images} test images, {n_test_labels} per-image LabelMe label files (Turkish diagnosis strings) -- inside test_data.zip, not extracted.')
    print('Not directly usable for evaluation -- needs a LabelMe-to-COCO conversion script (Phase 1/2 work, not done here).')
else:
    print('MISSING:', TEST_ZIP)

if any_missing:
    print()
    print('Some expected paths/members were missing -- inspect the zip contents directly, e.g.:')
    print('  zipfile.ZipFile(TRAINING_ZIP).namelist()[:50]')

# Free disk space: we've now read everything needed out of TRAINING_ZIP and
# TEST_ZIP above and don't need the zip files themselves anymore. Kaggle's
# /kaggle/working quota is small enough that leaving all three raw zips
# (11.8GB combined) on disk starves later steps (confirmed on a real run --
# even os.makedirs() for a new empty directory failed with "No space left on
# device" once the disk filled up completely).
for big_zip in (TRAINING_ZIP, TEST_ZIP):
    if os.path.exists(big_zip):
        size_gb = os.path.getsize(big_zip) / 1e9
        os.remove(big_zip)
        print(f'freed {size_gb:.1f}GB by removing {big_zip} (already read everything needed from it above)')
!df -h /kaggle/working

## Cell 3 -- single-image forward-pass smoke test

No pretrained HierarchicalDet weights exist publicly, so this uses the public
Swin-B ImageNet-22k backbone per `configs/diffdet.custom.swinbase.nonpretrain.yaml`,
with the detection head randomly initialized. The goal is only to prove the
model builds and runs a forward pass without crashing -- not to produce
meaningful detections (expect "detected 0 instances").

Verified locally end-to-end on 2026-07-14 (Python 3.12, CPU, a real
downloaded validation image) before this was ever run on Kaggle. This caught
a real bug, now fixed: `configs/diffdet.custom.swinbase.nonpretrain.yaml` had
`SWIN.SIZE: L-22k` (Large, embed_dim=192) while its own `MODEL.WEIGHTS`
pointed at a Base-sized checkpoint filename (embed_dim=128) -- an internal
inconsistency in the original authors' own config (the file is literally
named "swinbase"). This caused 325 silent shape-mismatch warnings and meant
the backbone was effectively randomly initialized -- easy to miss since
detectron2 logs shape mismatches as warnings, not errors. Fixed by
correcting `SIZE` to `B-22k`; confirmed zero shape mismatches after, with
only the expected non-backbone components (diffusion-process buffers,
detection head) reported as unmatched.

In [ ]:
# Download the Swin-B backbone (skipped if already present).
# IMPORTANT: keep the .pth extension -- detectron2's DetectionCheckpointer
# dispatches purely on file extension: ".pkl" is parsed as a Caffe2-style
# pickle blob, which would silently mis-load a native torch.save() file.
os.makedirs('models', exist_ok=True)
WEIGHTS_PATH = 'models/swin_base_patch4_window7_224_22k.pth'
if not os.path.exists(WEIGHTS_PATH):
    !wget -q -O $WEIGHTS_PATH https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_base_patch4_window7_224_22k.pth
print(WEIGHTS_PATH, os.path.getsize(WEIGHTS_PATH), 'bytes')

# Use a known-real validation image and the real category names
# (DENTEX_CATEGORY_JSON -> validation_triple.json) instead of a placeholder.
sample_image = os.path.join(VALIDATION_IMAGE_DIR, 'val_0.png')
assert os.path.exists(sample_image), f'{sample_image} not found -- check Cell 2 ran successfully.'
print('using sample image:', sample_image)

import subprocess
env = dict(os.environ, DENTEX_CATEGORY_JSON=VALIDATION_TRIPLE_PATH)
result = subprocess.run([
    sys.executable, 'demo.py',
    '--config-file', 'configs/diffdet.custom.swinbase.nonpretrain.yaml',
    '--input', sample_image,
    '--output', '/kaggle/working/smoke_test_output.jpg',
    '--opts', 'MODEL.WEIGHTS', WEIGHTS_PATH, 'MODEL.DEVICE', 'cuda',
], env=env)
assert result.returncode == 0, f'demo.py exited with code {result.returncode}'
print('Smoke test complete -- see /kaggle/working/smoke_test_output.jpg')